# 03 — Évaluation automatique des traductions (BLEU, chrF)

**Entrée :** `01_translations.csv` produit par le notebook 01. **Aucun modèle n'est chargé ici** :
ce notebook tourne en quelques secondes, sur le Mac comme sur Colab.

## Ce que mesurent les deux métriques

| Métrique | Principe | Points forts | Limites |
|---|---|---|---|
| **BLEU** (0–100) | Proportion de groupes de 1 à 4 **mots** de la traduction présents dans la référence, avec une pénalité si la traduction est trop courte | Standard historique, comparable entre publications | Un synonyme correct compte comme une erreur (*struggled* vs *had trouble*). Peu fiable sur peu de phrases |
| **chrF** (0–100) | Même idée, mais sur des groupes de **caractères** (F-score) | Récompense les mots proches (*complete* / *completed*). Plus stable sur un petit corpus | Ne juge ni la grammaire ni le naturel à l'oral |

**Règle :** ces scores mesurent la *ressemblance avec une référence*, pas la qualité absolue.
Nos références ont été rédigées par une seule personne : un modèle qui choisit d'autres mots
corrects sera pénalisé. Ils complètent l'analyse humaine, ils ne la remplacent pas.

## 1. Installation (inutile sur le Mac si `sacrebleu` est déjà dans le `.venv`)

In [ ]:
%pip install -q sacrebleu

## 2. Charger les résultats et le jeu de test

Le jeu de test sert à identifier les **morceaux de phrases coupées** (ids 18, 19, 21, 22) :
on les exclut des scores, car leurs références sont des fragments artificiels.
Ils restent analysés séparément dans le notebook 01.

In [ ]:
import csv
import json
from pathlib import Path

# Le premier chemin qui existe est utilisé : projet sur le Mac, ou fichier importé sur Colab.
RESULTS_CANDIDATES = [Path("../docs/experiments/01_cpu_2026-11-25/01_translations.csv"), Path("01_translations.csv")]
TEST_SET_CANDIDATES = [Path("translation_test_set.json")]


def first_existing(paths):
    for path in paths:
        if path.exists():
            return path
    raise FileNotFoundError(f"Aucun de ces fichiers n'existe : {[str(p) for p in paths]}")


results_path = first_existing(RESULTS_CANDIDATES)
test_set_path = first_existing(TEST_SET_CANDIDATES)

with results_path.open(encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
with test_set_path.open(encoding="utf-8") as f:
    test_set = json.load(f)

fragment_ids = {s["id"] for s in test_set["sentences"] if s.get("part", 0) > 0}

print("Résultats :", results_path, "-", len(rows), "lignes")
print("Morceaux exclus des scores :", sorted(fragment_ids))

## 3. Regrouper les traductions par modèle

In [ ]:
def group_by_model(rows, excluded_ids):
    """Retourne {modèle: [lignes]} en gardant l'ordre d'apparition des modèles."""
    by_model = {}
    for row in rows:
        if int(row["id"]) in excluded_ids:
            continue
        by_model.setdefault(row["model"], []).append(row)
    return by_model


by_model = group_by_model(rows, fragment_ids)

# Contrôle : chaque modèle doit avoir traduit exactement les mêmes phrases.
id_sets = {model: [int(r["id"]) for r in model_rows] for model, model_rows in by_model.items()}
if len({tuple(ids) for ids in id_sets.values()}) != 1:
    raise ValueError(f"Les modèles n'ont pas les mêmes phrases : {id_sets}")

print(len(by_model), "modèles,", len(next(iter(id_sets.values()))), "phrases évaluées par modèle")

## 4. Scores au niveau du corpus

- **BLEU / chrF** : score global sur toutes les phrases évaluées.
- **Ratio de longueur** : nombre de caractères de la traduction ÷ référence.
  Au-dessus de 1, la traduction est plus longue que la référence, ce qui est un indice utile pour le doublage.

In [ ]:
from sacrebleu.metrics import BLEU, CHRF

bleu_metric = BLEU()
chrf_metric = CHRF()

corpus_scores = []
for model, model_rows in by_model.items():
    hypotheses = [r["translation_en"] for r in model_rows]
    references = [r["reference_en"] for r in model_rows]

    corpus_scores.append({
        "model": model,
        "BLEU": round(bleu_metric.corpus_score(hypotheses, [references]).score, 1),
        "chrF": round(chrf_metric.corpus_score(hypotheses, [references]).score, 1),
        "length_ratio": round(sum(map(len, hypotheses)) / sum(map(len, references)), 2),
    })

print(f"{'Modèle':36} {'BLEU':>6} {'chrF':>6} {'Longueur':>9}")
for s in corpus_scores:
    print(f"{s['model']:36} {s['BLEU']:>6} {s['chrF']:>6} {s['length_ratio']:>9}")

print()
print("Signatures (pour la reproductibilité) :")
print(" ", bleu_metric.get_signature())
print(" ", chrf_metric.get_signature())

## 5. chrF phrase par phrase

Le score global cache les détails. Ici on voit **sur quelles phrases** les modèles se séparent.
Les scores d'une seule phrase sont très instables : ils servent à repérer des cas à relire, pas à conclure.

In [ ]:
models = list(by_model)
short_names = [m.split("/")[-1][:14] for m in models]

sentence_chrf = {}  # (modèle, id) -> score
for model, model_rows in by_model.items():
    for r in model_rows:
        score = chrf_metric.sentence_score(r["translation_en"], [r["reference_en"]]).score
        sentence_chrf[(model, int(r["id"]))] = round(score, 1)

print(f"{'id':>3} {'catégorie':18}" + "".join(f"{name:>16}" for name in short_names))
for sentence_id in id_sets[models[0]]:
    category = next(r["category"] for r in by_model[models[0]] if int(r["id"]) == sentence_id)
    scores = [sentence_chrf[(m, sentence_id)] for m in models]
    print(f"{sentence_id:>3} {category:18}" + "".join(f"{s:>16}" for s in scores))

## 6. chrF moyen par catégorie de difficulté

In [ ]:
categories = []
for r in by_model[models[0]]:
    if r["category"] not in categories:
        categories.append(r["category"])

print(f"{'catégorie':18}" + "".join(f"{name:>16}" for name in short_names))
for category in categories:
    line = f"{category:18}"
    for model in models:
        scores = [sentence_chrf[(model, int(r["id"]))] for r in by_model[model] if r["category"] == category]
        line += f"{sum(scores) / len(scores):>16.1f}"
    print(line)

## 7. Sauvegarder les scores à côté des résultats de l'expérience

In [ ]:
output_path = results_path.parent / "03_scores.csv"

with output_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["model", "BLEU", "chrF", "length_ratio"])
    writer.writeheader()
    writer.writerows(corpus_scores)

print("Écrit :", output_path)